# 🚬 煙霧偵測模型訓練 — Google Colab

**資料集：** [Roboflow smoke-gxoy3](https://universe.roboflow.com/naruesuan-university/smoke-gxoy3)  
**模型：** YOLOv8n（輕量，適合邊緣部署）  
**目標：** 訓練後匯出 ONNX 模型，複製到 `models/smoke_detector.onnx` 啟用條件 3

## 開始前請確認
- 上方選單：`執行階段 → 變更執行階段類型 → GPU (T4)`
- 準備好 Roboflow API Key（[取得方式](https://app.roboflow.com/) → Settings → Roboflow API）

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. 安裝套件

In [ ]:
!pip install -q roboflow ultralytics onnx onnxsim

## 2. 下載煙霧資料集

輸入你的 Roboflow API Key：

In [ ]:
from roboflow import Roboflow

API_KEY = "YOUR_ROBOFLOW_API_KEY"  # ← 替換為你的 Key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("naruesuan-university").project("smoke-gxoy3")
version = project.version(3)
dataset = version.download("yolov8", location="/content/smoke_dataset")
print(f"資料集路徑：{dataset.location}")

## 3. 確認資料集結構與統計

In [ ]:
import os, glob

base = "/content/smoke_dataset"

# 自動偵測目錄結構
if os.path.exists(f"{base}/images/train"):
    train_dir = f"{base}/images/train"
    val_dir   = f"{base}/images/val"
elif os.path.exists(f"{base}/train/images"):
    train_dir = f"{base}/train/images"
    val_dir   = f"{base}/valid/images"
else:
    # 搜尋 data.yaml 中的路徑
    import yaml
    with open(f"{base}/data.yaml") as f:
        cfg = yaml.safe_load(f)
    print("data.yaml 內容:", cfg)
    train_dir = val_dir = None

if train_dir:
    train_imgs = glob.glob(f"{train_dir}/*.jpg") + glob.glob(f"{train_dir}/*.png")
    val_imgs   = glob.glob(f"{val_dir}/*.jpg")   + glob.glob(f"{val_dir}/*.png")
    print(f"✅ Train: {len(train_imgs)} 張")
    print(f"✅ Val  : {len(val_imgs)} 張")

# 列出 data.yaml
!cat /content/smoke_dataset/data.yaml

## 4. 修正 data.yaml（確保路徑正確）

In [ ]:
import yaml, os

yaml_path = "/content/smoke_dataset/data.yaml"
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

# 統一修正為絕對路徑
cfg["path"] = "/content/smoke_dataset"

# 相容 train/valid 或 images/train 兩種結構
if os.path.exists("/content/smoke_dataset/train/images"):
    cfg["train"] = "train/images"
    cfg["val"]   = "valid/images"
else:
    cfg["train"] = "images/train"
    cfg["val"]   = "images/val"

cfg["nc"]    = 1
cfg["names"] = ["smoke"]

with open(yaml_path, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print("修正後的 data.yaml：")
!cat /content/smoke_dataset/data.yaml

## 5. 訓練模型

T4 GPU 大約需要 **20–40 分鐘**（100 epochs, batch=16）

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # 下載預訓練權重（約 6MB）

results = model.train(
    data="/content/smoke_dataset/data.yaml",
    epochs=100,
    batch=16,
    imgsz=640,
    device=0,                 # GPU
    project="/content/runs",
    name="smoke_detector",
    patience=20,              # Early stopping
    # 資料增強（適合煙霧形態多變的特性）
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    save=True,
    save_period=10,
    exist_ok=True,
)

## 6. 查看訓練結果

In [ ]:
from IPython.display import Image as IPImage, display
import glob

# 訓練曲線
plots = glob.glob("/content/runs/smoke_detector/*.png")
for p in sorted(plots):
    display(IPImage(p))

In [ ]:
# 驗證最終指標
best_pt = "/content/runs/smoke_detector/weights/best.pt"
eval_model = YOLO(best_pt)
metrics = eval_model.val(data="/content/smoke_dataset/data.yaml")
print(f"mAP50   : {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall  : {metrics.box.mr:.4f}")

## 7. 匯出 ONNX 模型

In [ ]:
export_model = YOLO(best_pt)
export_model.export(
    format="onnx",
    imgsz=640,
    simplify=True,   # onnxsim 簡化計算圖，加速推論
    opset=17,
)

onnx_path = best_pt.replace(".pt", ".onnx")
print(f"\n✅ ONNX 模型位於：{onnx_path}")

## 8. 下載模型到本機

執行後會自動下載 `smoke_detector.onnx`，下載完成後複製到專案的 `models/` 目錄：

In [ ]:
import shutil
from google.colab import files

# 複製為統一命名
dst = "/content/smoke_detector.onnx"
shutil.copy(onnx_path, dst)
print(f"檔案大小：{os.path.getsize(dst)/1024/1024:.1f} MB")

# 下載到本機
files.download(dst)
print("\n📦 下載完成後，將 smoke_detector.onnx 複製到專案的 models/ 目錄")
print("   路徑：cgr_detection/models/smoke_detector.onnx")
print("   重啟推論系統即自動啟用條件 3（煙霧偵測）")

## 9. （選用）同時下載 .pt 權重備份

In [ ]:
# 如果想保留 PyTorch 權重以便日後繼續訓練
files.download(best_pt)

---

## 完成後的部署步驟

```bash
# 1. 將下載的模型放到專案目錄
cp ~/Downloads/smoke_detector.onnx /path/to/cgr_detection/models/

# 2. 重啟推論系統（自動偵測並啟用條件 3）
python infer_main.py
```

啟動時會看到：
```
[func] 煙霧偵測模型已啟用（條件 3）
```

條件 3 觸發時畫面顯示橘色煙霧框與 `[+10 Smoke]` 計分提示。